# Train on ZINC in memory

**Purpose:** Train on ZINC in memory.

**Before you start:** NSPPK, RDKit, and a ZINC CSV; sufficient RAM for the selected data. Use the Python environment prepared by [setup](../setup.ipynb).

**Results:** Model and training progress artifacts.

Run the cells in order, reviewing the configuration before starting the main work. Data stays under `notebooks/datasets`; models and outputs use the project’s artifact folders.


Load the training tools and locate data and output folders.


In [ ]:
print('Load the training tools and locate data and output folders.')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

from pathlib import Path
import random
import warnings

import numpy as np
from IPython.core.display import HTML

HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell; text-align: center; vertical-align: middle;}</style>')

warnings.filterwarnings('ignore', message=r".*`isinstance\(treespec, LeafSpec\)` is deprecated.*")

from conditional_node_field_graph_generator.notebooks import configure_notebook, download_zinc_dataset
globals().update(configure_notebook(require_nsppk=True, print_torch=True))
import torch
torch.set_float32_matmul_precision('high')

from abstractgraph_graphicalizer.chem import draw_molecule, draw_molecules
from conditional_node_field_graph_generator.input_sources import iter_selected_source_graphs
from conditional_node_field_graph_generator.extensions.demo import show_molecules
from conditional_node_field_graph_generator.extensions.demo.visualization import plot_networkx_graphs
from conditional_node_field_graph_generator.extensions.demo.pipeline import build_graph_generator


Choose the dataset, model size, training budget, and decoder settings.


In [ ]:
print('Choose the dataset, model size, training budget, and decoder settings.')
ZINC_DATA_ROOT = NOTEBOOK_DATA_ROOT / 'zinc'
ZINC_SIZE = 'zinc'
ZINC_FILENAME = f'{ZINC_SIZE}.csv'
RANDOM_SEED = 42
# zinc.csv has roughly 499k rows. Use 200k for the first larger run;
# set DATA_LIMIT = None to consume the full file.
DATA_LIMIT = None
# The all-pairs edge-count loss scales as batch * max_nodes^2 * latent_dim.
# Batch 256 exceeds a 16 GB-class GPU; 96 keeps the d512 model practical.
FIT_BATCH_SIZE = 96
MAXIMUM_EPOCHS = 350
LOSS_CURVES_PDF_EVERY_N_EPOCHS = 10

# Fit SVD bases on a fixed row sample, then project the full dataset.
EMBEDDING_DIM = 512
EMBEDDING_SVD_DIM = 192
EMBEDDING_SVD_FIT_MAX_ROWS = 100_000
EMBEDDING_SVD_TRANSFORM_BATCH_SIZE = 10_000
LEARNING_RATE = 1e-4
SPARSE_SUPERVISION_MASK_RATIO = 0.35

# Capacity knobs. The full-data d384/l8 run still showed no train/validation divergence,
# so increase representation and depth while easing dropout.
NUMBER_OF_TRANSFORMER_LAYERS = 10
TRANSFORMER_ATTENTION_HEAD_COUNT = 8
TRANSFORMER_DROPOUT = 0.05

# Structural supervision weights.
LAMBDA_DEGREE_IMPORTANCE = 4.0
LAMBDA_NODE_LABEL_IMPORTANCE = 3.0
LAMBDA_EDGE_LABEL_IMPORTANCE = 3.0
LAMBDA_DIRECT_EDGE_IMPORTANCE = 3.0
LAMBDA_AUXILIARY_EDGE_IMPORTANCE = 2.0

DATA_LIMIT_LABEL = 'all' if DATA_LIMIT is None else DATA_LIMIT
MODEL_NAME = (
    f'{ZINC_SIZE}-d{EMBEDDING_DIM}-svd{EMBEDDING_SVD_DIM}-s{DATA_LIMIT_LABEL}'
    f'-b{FIT_BATCH_SIZE}-lr{LEARNING_RATE:g}-mask{SPARSE_SUPERVISION_MASK_RATIO:g}'
    f'-l{NUMBER_OF_TRANSFORMER_LAYERS}-h{TRANSFORMER_ATTENTION_HEAD_COUNT}'
    f'-drop{TRANSFORMER_DROPOUT:g}'
    f'-e{MAXIMUM_EPOCHS}'
)
# Parallel preprocessing is enabled below. Set both parallel flags to False
# and FEASIBILITY_N_JOBS to 1 if preprocessing stalls in this kernel.
VECTORIZER_PARALLEL = True
FEASIBILITY_PARALLEL = True
FEASIBILITY_N_JOBS = -1
DECODER_N_JOBS = -1
DECODER_SOLVER_THREADS = 16
MAX_DECODE_SECONDS_PER_SAMPLE = 30.0
MAX_DECODE_ATTEMPTS_PER_SAMPLE = 2


random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


Load the selected ZINC graphs into memory and build the generator.


In [ ]:
print('Load the selected ZINC graphs into memory and build the generator.')
csv_path = download_zinc_dataset(ZINC_DATA_ROOT, filename=ZINC_FILENAME)
print(f'ZINC CSV: {csv_path}')

graphs = list(
    iter_selected_source_graphs(
        csv_path,
        'zinc_csv',
        limit=DATA_LIMIT,
        random_state=RANDOM_SEED,
    )
)
print(f'Loaded {len(graphs)} graphs into memory for non-streaming fit.')

# Recurrent energy setup. Set node_field_mode to "baseline" for the original model.
NODE_FIELD_CONFIG = {
    "node_field_mode": "recurrent_energy",
    "recurrent_hidden_dimension": None,  # None uses latent_embedding_dimension.
    "recurrent_training_steps": 8,
    "recurrent_detach_interval": 4,  # None keeps full backpropagation through memory.
    "recurrent_update_scale": 1.0,
    "recurrent_initial_state": "zeros",
    "recurrent_state_normalization": True,
    "recurrent_corruption_schedule": "annealed",  # Also "constant" or "none".
    "recurrent_sigma_min": 0.02,
    "recurrent_sigma_max": None,  # None uses node_field_sigma.
    "recurrent_supervise_all_steps": True,
    "recurrent_loss_discount": 1.0,
}

# Keep new model artifacts distinct from existing baseline checkpoints.
MODEL_NAME = (
    MODEL_NAME.removesuffix("-baseline").removesuffix("-recurrent_energy")
    + "-" + NODE_FIELD_CONFIG["node_field_mode"]
)

graph_generator = build_graph_generator(
    **NODE_FIELD_CONFIG,
    latent_embedding_dimension=EMBEDDING_DIM,
    node_embedding_svd_dimension=EMBEDDING_SVD_DIM,
    graph_embedding_svd_dimension=EMBEDDING_SVD_DIM,
    embedding_svd_fit_max_rows=EMBEDDING_SVD_FIT_MAX_ROWS,
    embedding_svd_transform_batch_size=EMBEDDING_SVD_TRANSFORM_BATCH_SIZE,
    number_of_transformer_layers=NUMBER_OF_TRANSFORMER_LAYERS,
    transformer_attention_head_count=TRANSFORMER_ATTENTION_HEAD_COUNT,
    transformer_dropout=TRANSFORMER_DROPOUT,
    node_vectorizer_dense=False,
    graph_vectorizer_dense=False,
    node_vectorizer_parallel=VECTORIZER_PARALLEL,
    graph_vectorizer_parallel=VECTORIZER_PARALLEL,
    feasibility_parallel=FEASIBILITY_PARALLEL,
    feasibility_n_jobs=FEASIBILITY_N_JOBS,
    locality_horizon=2,
    sparse_supervision_mask_ratio=SPARSE_SUPERVISION_MASK_RATIO,
    learning_rate=LEARNING_RATE,
    lambda_degree_importance=LAMBDA_DEGREE_IMPORTANCE,
    lambda_node_label_importance=LAMBDA_NODE_LABEL_IMPORTANCE,
    lambda_edge_label_importance=LAMBDA_EDGE_LABEL_IMPORTANCE,
    lambda_direct_edge_importance=LAMBDA_DIRECT_EDGE_IMPORTANCE,
    lambda_auxiliary_edge_importance=LAMBDA_AUXILIARY_EDGE_IMPORTANCE,
    maximum_epochs=MAXIMUM_EPOCHS,
    batch_size=FIT_BATCH_SIZE,
    verbose=1,
    stream_snapshot_every_n_batches=5,
    decoder_n_jobs=DECODER_N_JOBS,
    decoder_solver_threads=DECODER_SOLVER_THREADS,
    max_decode_seconds_per_sample=MAX_DECODE_SECONDS_PER_SAMPLE,
    max_decode_attempts_per_sample=MAX_DECODE_ATTEMPTS_PER_SAMPLE,
    artifact_root=ARTIFACT_ROOT,
    checkpoint_root=CHECKPOINT_ROOT,
    model_name=MODEL_NAME,
    model_dir=SAVED_GENERATOR_ROOT,
)
print('node_vectorizer_dense =', graph_generator.node_graph_vectorizer.dense)
print('graph_vectorizer_dense =', graph_generator.graph_vectorizer.dense)
graph_generator.loss_curves_pdf_every_n_epochs = LOSS_CURVES_PDF_EVERY_N_EPOCHS
graph_generator.graph_decoder.diagnostic_graph_renderer = draw_molecules
graph_generator.graph_decoder.adjacency_time_limit_seconds = MAX_DECODE_SECONDS_PER_SAMPLE
graph_generator.graph_decoder.parallel_decode_timeout_seconds = MAX_DECODE_SECONDS_PER_SAMPLE

TRAINING_PROGRESS_MOLECULE_PLOT_KWARGS = {
    'size': (500, 350),
    'cell_size': 2.8,
    'title_font_size': 8,
}
TRAINING_PROGRESS_PDF_PATH = ARTIFACT_ROOT / 'samples' / MODEL_NAME / 'training_samples.pdf'
print(f'Training sample PDF: {TRAINING_PROGRESS_PDF_PATH}')


Train the generator and write training progress artifacts. This may take a long time.


In [ ]:
%%time
print('Train the generator and write training progress artifacts. This may take a long time.')
graph_generator.fit(
    graphs,
    train_node_generator=True,
    targets=None,
    sample_training_progress=True,
    sample_training_progress_n_samples=1,
    sample_training_progress_every_n_epochs=5,
    sample_training_progress_pdf_path=TRAINING_PROGRESS_PDF_PATH,
    sample_training_progress_plot_kwargs=TRAINING_PROGRESS_MOLECULE_PLOT_KWARGS,
    sample_training_progress_plot_fn=draw_molecule,
)

print('training_graph_conditioning_ =', len(graph_generator.training_graph_conditioning_))
print('is_fitted_ =', graph_generator.is_fitted_)


Optionally reload a saved model; leave reloading disabled to inspect the model just trained.


In [ ]:
print('Optionally reload a saved model; leave reloading disabled to inspect the model just trained.')
#load model
from conditional_node_field_graph_generator.persistence import (
    list_saved_graph_generators,
    load_graph_generator,
)
SAVED_GENERATOR_ROOT = REPO_ROOT / '.artifacts' / 'saved_generators'
list_saved_graph_generators(SAVED_GENERATOR_ROOT)
#MODEL_NAME = 'zinc15-nonstreaming-d64-s10000-b128-e350'
RELOAD_SAVED_MODEL = False
if RELOAD_SAVED_MODEL:
    graph_generator = load_graph_generator(MODEL_NAME+'.pkl', model_dir=SAVED_GENERATOR_ROOT)
    graph_generator.graph_decoder.solver_threads = DECODER_SOLVER_THREADS
    graph_generator.max_decode_seconds_per_sample = MAX_DECODE_SECONDS_PER_SAMPLE
    graph_generator.max_decode_attempts_per_sample = MAX_DECODE_ATTEMPTS_PER_SAMPLE
    graph_generator.graph_decoder.adjacency_time_limit_seconds = MAX_DECODE_SECONDS_PER_SAMPLE
    graph_generator.graph_decoder.parallel_decode_timeout_seconds = MAX_DECODE_SECONDS_PER_SAMPLE


Generate molecules and compare the decoding stages.


In [ ]:
print('Generate molecules and compare the decoding stages.')
n_samples = 3
FEASIBILITY_EFFORT = 5
FEASIBILITY_FILTER = 'strict'  # 'none', 'fallback', or 'strict'

sample_variants = graph_generator.sample(
    n_samples=n_samples,
    feasibility_effort=FEASIBILITY_EFFORT,
    return_decode_stages=True,
)
effort_keys = sorted(sample_variants, key=lambda key: int(key.split('_')[1]))

for variant_key in effort_keys:
    effort = int(variant_key.split('_')[1])
    title_prefix = f'effort {effort}'
    variant_samples = [graph for graph in sample_variants[variant_key] if graph is not None]
    if not variant_samples:
        print(f'No {title_prefix} samples to display.')
        continue
    show_molecules(
        variant_samples,
        n=len(variant_samples),
        title=f'Non-streaming ZINC {title_prefix} samples',
        legends=[f'{title_prefix} {idx}' for idx in range(len(variant_samples))],
        n_graphs_per_line=n_samples,
    )


Generate and display molecules with the selected feasibility filter.


In [ ]:
print('Generate and display molecules with the selected feasibility filter.')
if graph_generator.feasibility_estimator is None and FEASIBILITY_FILTER != 'none':
    raise RuntimeError('Feasibility estimator is unavailable in this environment.')

filtered_samples = graph_generator.sample(
    n_samples=n_samples,
    feasibility_effort=FEASIBILITY_EFFORT,
    feasibility_filter=FEASIBILITY_FILTER,
)
show_molecules(
    filtered_samples,
    n=len(filtered_samples),
    title=f'Non-streaming ZINC samples | effort={FEASIBILITY_EFFORT}, filter={FEASIBILITY_FILTER}',
    n_graphs_per_line=max(1, n_samples),
)
